In [ ]:
!pip install transformers torch

In [ ]:
# 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import os
import ast  # CSV에서 문자열형 리스트를 실제 리스트로 변환하기 위해 필요
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, save_npz
from tqdm import tqdm  # 진행 상황 확인용

# ----------------------------------------------------
# ⚠️ BERT 관련 라이브러리 (Colab에서 설치 필수)
# !pip install transformers torch
# ----------------------------------------------------

try:
    import torch
    from transformers import AutoTokenizer, AutoModel

    # GPU (CUDA) 사용 여부 확인 및 장치 설정
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"✅ PyTorch Device Set: {DEVICE}")

except ImportError:
    print("FATAL ERROR: torch 또는 transformers 라이브러리가 설치되지 않았습니다.")
    print("Colab에서 !pip install transformers torch 를 실행하세요.")
    exit()

# ----------------------------------------------------
# 1. BERT 임베딩 함수 정의 (다국어 모델)
# ----------------------------------------------------
def get_bert_embeddings(texts, model_name='bert-base-multilingual-cased', max_len=256):
    """
    다국어 BERT 모델을 사용하여 텍스트 리스트의 [CLS] 토큰 임베딩을 추출합니다.
    """
    print(f"Loading Multilingual Model: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(DEVICE)
    model.eval()  # 추론 모드 설정

    all_embeddings = []

    # 데이터를 배치(Batch) 단위로 처리
    BATCH_SIZE = 32

    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="BERT Embedding"):
        batch_texts = texts[i:i + BATCH_SIZE]

        # 텍스트를 토큰화 및 패딩
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_len
        ).to(DEVICE)

        with torch.no_grad():
            outputs = model(**inputs)

        # [CLS] 토큰 벡터 추출
        embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(embeddings)

    return np.vstack(all_embeddings)

# ----------------------------------------------------
# 2. 메인 실행 파이프라인
# ----------------------------------------------------

# (입력) 이전 단계에서 전처리한 파일
GAME_FILE = '/content/drive/MyDrive/2025Bigdata/game/preprocessed_steam_data.csv'

# (출력) 저장될 파일 이름
OUTPUT_EMBEDDING_FILE = 'game_embeddings.npz'
OUTPUT_ID_FILE = 'embedding_appids.csv'

# (설정) is_free=True일 때 부여할 가중치 값
FREE_GAME_WEIGHT = 5.0

try:
    print(f"Loading preprocessed data: {GAME_FILE}")
    game_df = pd.read_csv(GAME_FILE)

    # --- 2.1. 피처 엔지니어링 ---
    print("Feature Engineering 시작...")

    # 2.1.1. (Text) 줄거리 (BERT 입력용)
    # (!!! 수정 !!!) 요청대로 'short_description'만 BERT 입력으로 사용합니다.
    game_df['short_description'] = game_df['short_description'].fillna("")
    game_df['text_for_embedding'] = game_df['short_description']

    # 2.1.2. (Categorical) Tags (TF-IDF 입력용)
    # (!!! 수정 !!!) 'genres'를 제외하고 'tags'만 처리합니다.
    # CSV로 저장된 "['a', 'b']" 같은 문자열을 실제 리스트 ['a', 'b']로 변환
    game_df['tags'] = game_df['tags'].fillna('[]').apply(ast.literal_eval)

    # TF-IDF가 인식할 수 있도록 리스트를 공백으로 구분된 문자열로 변환
    game_df['tags_str'] = game_df['tags'].apply(lambda x: ' '.join(x))

    # 2.1.3. (Numerical) is_free 가중치
    game_df['is_free'] = game_df['is_free'].fillna(False)
    is_free_vectors = game_df['is_free'].apply(
        lambda x: FREE_GAME_WEIGHT if x else 0.0
    ).values.reshape(-1, 1)  # hstack을 위해 (N, 1) 형태로 reshape

    print("Feature Engineering 완료.")

    # --- 2.2. 벡터화 (Embedding) ---

    # 2.2.1. Tags (TF-IDF)
    # (!!! 수정 !!!) 'genres' 관련 TF-IDF 코드 삭제
    print("TF-IDF 벡터화 (Tags Only) 시작...")
    tfidf_tags = TfidfVectorizer(max_features=500)  # Tags 피처 수 (조절 가능)

    tfidf_tags_vectors = tfidf_tags.fit_transform(game_df['tags_str'])
    print(f"Tags TF-IDF shape: {tfidf_tags_vectors.shape}")

    # 2.2.2. Text (Multilingual BERT)
    print("BERT 임베딩 (Description Only) 시작...")
    # (!!! 수정 !!!) 'text_for_embedding' (줄거리만 담긴) 컬럼을 전달
    bert_embeddings = get_bert_embeddings(
        game_df['text_for_embedding'].tolist(),
        model_name='bert-base-multilingual-cased'
    )
    print(f"BERT 임베딩 완료. (shape: {bert_embeddings.shape})")

    # --- 2.3. 모든 피처 결합 (hstack) ---
    print("모든 피처 벡터 결합 중... (hstack)")

    # (!!! 수정 !!!) 'tfidf_genres_vectors'를 결합 리스트에서 제외
    # [ (N, 768) BERT ] + [ (N, 500) Tags ] + [ (N, 1) is_free ]
    combined_features = hstack([
        bert_embeddings,
        tfidf_tags_vectors,
        is_free_vectors
    ], format='csr')  # CSR(Compressed Sparse Row) 포맷으로 효율적으로 결합

    print(f"결합 완료. 최종 임베딩 행렬 shape: {combined_features.shape}")

    # --- 2.4. 결과 저장 ---

    # 1. 최종 임베딩 희소 행렬 저장
    print(f"'{OUTPUT_EMBEDDING_FILE}' 파일로 임베딩 저장 중...")
    save_npz(OUTPUT_EMBEDDING_FILE, combined_features)

    # 2. 임베딩 순서(행)에 맞는 appid 저장 (매핑용)
    print(f"'{OUTPUT_ID_FILE}' 파일로 appid 매핑 저장 중...")
    game_df['appid'].to_csv(OUTPUT_ID_FILE, index=False, header=True)

    print("\n--- 모든 작업 완료 ---")
    print(f"'{OUTPUT_EMBEDDING_FILE}'와 '{OUTPUT_ID_FILE}'이 생성되었습니다.")

except FileNotFoundError:
    print(f"--- 작업 실패 ---")
    print(f"오류: '{GAME_FILE}' 파일을 찾을 수 없습니다.")
    print("이전 단계의 전처리 코드가 먼저 실행되었는지, 파일 이름이 맞는지 확인하세요.")
except Exception as e:
    print(f"--- 작업 실패 ---")
    print(f"임베딩 중 오류가 발생했습니다: {e}")

✅ PyTorch Device Set: cuda
Loading preprocessed data: /content/drive/MyDrive/2025Bigdata/game/preprocessed_steam_data.csv
Feature Engineering 시작...
Feature Engineering 완료.
TF-IDF 벡터화 (Tags Only) 시작...
Tags TF-IDF shape: (4507, 480)
BERT 임베딩 (Description Only) 시작...
Loading Multilingual Model: bert-base-multilingual-cased


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

BERT Embedding: 100%|██████████| 141/141 [00:19<00:00,  7.40it/s]


BERT 임베딩 완료. (shape: (4507, 768))
모든 피처 벡터 결합 중... (hstack)
결합 완료. 최종 임베딩 행렬 shape: (4507, 1249)
'game_embeddings.npz' 파일로 임베딩 저장 중...
'embedding_appids.csv' 파일로 appid 매핑 저장 중...

--- 모든 작업 완료 ---
'game_embeddings.npz'와 'embedding_appids.csv'이 생성되었습니다.
